# 📘 Guía de Estudio 1: Entendiendo los Landmarks de MediaPipe y su Normalización

En esta primera guía interactiva, aprenderás:
1. Cómo están estructurados los datos de entrada (landmarks 3D).
2. Por qué necesitamos la **Normalización Multi-Stream**.
3. Cómo manipular y visualizar estos datos paso a paso en PyTorch.

---

### 1. Concepto Base: ¿Qué es un Landmark?

Un **landmark** es un punto clave en el espacio 3D ($x, y, z$) detectado por MediaPipe Holistic. Representa articulaciones, puntos del rostro o partes del cuerpo.

En nuestro proyecto, procesamos **4 streams (flujos) independientes**:
- **Mano Izquierda (Left Hand):** 21 landmarks ($21 \times 3$ coordenadas).
- **Mano Derecha (Right Hand):** 21 landmarks ($21 \times 3$ coordenadas).
- **Rostro (Face):** 468 landmarks ($468 \times 3$ coordenadas).
- **Pose (Cuerpo/Hombros):** 33 landmarks ($33 \times 3$ coordenadas).

Cuando guardamos una seña completa de $T$ frames (imágenes), guardamos un diccionario con 4 tensores de dimensiones:
- `left_hand`: $[T, 21, 3]$
- `right_hand`: $[T, 21, 3]$
- `face`: $[T, 468, 3]$
- `pose`: $[T, 33, 3]$

In [1]:
import torch
import os
import glob

# Vamos a buscar un archivo de landmarks real de nuestro dataset procesado
data_dir = "../../data/processed/landmarks"
file_paths = glob.glob(os.path.join(data_dir, "**/*.pt"), recursive=True)

if len(file_paths) == 0:
    print("[-] ¡No se encontraron archivos procesados! Asegúrate de haber ejecutado extract_landmarks.py")
else:
    sample_path = file_paths[0]
    print(f"[+] Cargando archivo de ejemplo: {sample_path}")
    landmarks = torch.load(sample_path, weights_only=True)
    
    # Mostrar las dimensiones
    for stream_name, tensor in landmarks.items():
        print(f"Stream: {stream_name:<12} | Dimensiones: {list(tensor.shape)}")

[+] Cargando archivo de ejemplo: ../../data/processed/landmarks/LSC70/LSC70AN/Per01/1.pt
Stream: left_hand    | Dimensiones: [6, 21, 3]
Stream: right_hand   | Dimensiones: [6, 21, 3]
Stream: face         | Dimensiones: [6, 468, 3]
Stream: pose         | Dimensiones: [6, 33, 3]


### 2. ¿Por qué normalizar? (La matemática explicada)

Si una persona firma una seña parada cerca de la cámara vs. lejos de la cámara, los números de las coordenadas de MediaPipe cambian drásticamente. Lo mismo pasa si se mueve a la izquierda o derecha.

Para lograr **invarianza espacial** (que el modelo entienda el gesto sin importar la distancia o posición de la cámara), aplicamos normalización local por stream:

#### 🖐️ Normalización de Mano
1. **Centrado:** Restamos la coordenada de la muñeca (Landmark 0) a todos los demás puntos de la mano. Así, la muñeca se vuelve el origen $(0,0,0)$.
2. **Escalado:** Calculamos la distancia euclidiana entre la muñeca (0) y el nudillo del dedo medio (9), y dividimos todas las coordenadas por esta distancia. Esto hace que si la mano está cerca o lejos, siempre tenga el mismo tamaño normalizado.

$$\mathbf{p}_{norm} = \frac{\mathbf{p} - \mathbf{p}_{wrist}}{\|\mathbf{p}_{wrist} - \mathbf{p}_{middle}\| + \epsilon}$$

### 📝 Ejercicio 1: Implementar normalización de mano de forma manual

A continuación, tienes un tensor de mano artificial con 21 landmarks sin normalizar. Completa la función de normalización y valida que la muñeca (índice 0) quede exactamente en `[0.0, 0.0, 0.0]` y que las dimensiones se mantengan.

In [ ]:
import numpy as np

# 21 landmarks aleatorios simulando coordenadas reales de una mano frente a la cámara
mano_sucia = np.random.rand(21, 3) * 10.0 + 5.0 # valores entre 5.0 y 15.0

def normalizar_mano_ejercicio(coords):
    """
    Coords: numpy array de shape (21, 3)
    """
    # TODO: Escribe tu código aquí
    # 1. Obtén la muñeca (coords[0])
    # 2. Obtén el nudillo del dedo medio (coords[9])
    # 3. Calcula la distancia euclidiana entre muñeca y nudillo
    # 4. Resta la muñeca de coords y divide por la distancia (añade 1e-8 para evitar dividir por 0)
    muñeca = coords[0]
    nudillo = coords[9]
    d = np.sqrd(())

# --- Validación ---
mano_limpia = normalizar_mano_ejercicio(mano_sucia)
if mano_limpia is not None:
    print(f"Posición de la muñeca normalizada (debe ser [0,0,0]): {mano_limpia[0]}")
    print(f"Distancia normalizada muñeca-dedo medio (debe ser 1.0): {np.linalg.norm(mano_limpia[0] - mano_limpia[9]):.4f}")
else:
    print("[-] Por favor, completa la función primero.")

### Solución sugerida para validar el ejercicio:
```python
def normalizar_mano_ejercicio(coords):
    wrist = coords[0]
    middle = coords[9]
    scale = np.linalg.norm(wrist - middle) + 1e-8
    return (coords - wrist) / scale
```